In [1]:
# IMPORTS

import pandas as pd
import os


In [2]:
# PATHS

path_raw = os.path.join('..', 'data', 'raw')

path_processed = os.path.join('..', 'data', 'processed')

os.makedirs(path_processed, exist_ok=True)


In [3]:
# FILE PATHS

file_clock = os.path.join(path_raw, 'raw_africa_clock_15_35.xlsx')

file_edu_unemp = os.path.join(path_raw, 'Data_africa_education_unemployed.csv')

file_sector = os.path.join(path_raw, 'Data_africa_sector_employed.csv')



In [4]:
# LOAD DATASETS

df_clock = pd.read_excel(file_clock)

df_edu_unemp = pd.read_csv(file_edu_unemp)

df_sector = pd.read_csv(file_sector)


In [5]:
# COLUMN HARMONIZATION

mapping_cols = {
    'ccode': 'code_pays',
    'country': 'pays',
    'year': 'annee',
    'gender': 'genre',
    'age': 'age_valeur',
    'age_group': 'tranche_age'
}

df_clock.rename(columns=mapping_cols, inplace=True)

df_edu_unemp.rename(columns=mapping_cols, inplace=True)

df_sector.rename(columns=mapping_cols, inplace=True)

In [6]:
# NORMALIZATION

for df in [df_clock, df_edu_unemp, df_sector]:

    df['genre'] = df['genre'].astype(str).str.lower()

    df['annee'] = df['annee'].astype(int)


In [7]:
# AGE GROUP CREATION

def regrouper_age(age):

    if 15 <= age <= 24:
        return '15-24'

    if 25 <= age <= 35:
        return '25-35'

    return '35+'


df_clock['tranche_age'] = df_clock['age_valeur'].apply(regrouper_age)

df_sector['tranche_age'] = df_sector['age_valeur'].apply(regrouper_age)


In [8]:
# FILTER CÔTE D'IVOIRE

codes_ci = ['CIV', "Cote d'Ivoire", "Ivory Coast"]


df_ci_sector = df_sector[
    (
        df_sector['code_pays'].isin(codes_ci)
    )
    &
    (
        df_sector['tranche_age'] != '35+'
    )
].copy()


df_ci_edu = df_edu_unemp[
    df_edu_unemp['code_pays'].isin(codes_ci)
].copy()


df_ci_clock = df_clock[
    (
        df_clock['code_pays'].isin(codes_ci)
    )
].copy()

In [9]:
# CHECK FILTER RESULTS

print("Sector dataset shape:", df_ci_sector.shape)

print("Education dataset shape:", df_ci_edu.shape)

print("Clock dataset shape:", df_ci_clock.shape)


Sector dataset shape: (9702, 8)
Education dataset shape: (330, 7)
Clock dataset shape: (882, 9)


In [10]:
# SAVE PROCESSED DATASETS

sector_path = os.path.join(
    path_processed,
    'ci_sector_15_35.csv'
)

education_path = os.path.join(
    path_processed,
    'ci_education_15_35.csv'
)

clock_path = os.path.join(
    path_processed,
    'ci_clock_snapshot_2026.csv'
)


df_ci_sector.to_csv(sector_path, index=False)

df_ci_edu.to_csv(education_path, index=False)

df_ci_clock.to_csv(clock_path, index=False)

In [11]:
# FINAL CHECK

print("\nProcessed files created successfully:\n")

for file in [
    sector_path,
    education_path,
    clock_path
]:

    print(file)


Processed files created successfully:

../data/processed/ci_sector_15_35.csv
../data/processed/ci_education_15_35.csv
../data/processed/ci_clock_snapshot_2026.csv
